# Suite CTY — The C Types dialect

`Expressions.C.Types` models C's type names (ISO/IEC 9899, 6.7.7 in C11, "Type names"), a grammar of their own, as
mbse-programs has them: a type name's specifiers and its abstract declarator, and an enumeration's body, its enumerators
and the directives and comments among them, as data. C's terms hold its terms where a type stands (a cast's, a compound
literal's, a generic association's type) and where an expression or a type may (`sizeof(int)`), C reading them. Its
kinds are validated, bind nothing, cover C's three editions, print as mbse-programs prints them, and are refused by
evaluation by name; a cast reads its type from it.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as CFamily from "@mbse/expressions/C/Expressions";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor";
import * as CTypes from "@mbse/expressions/C/Types";
import * as CppFamily from "@mbse/expressions/Cpp/Expressions";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import * as AMD64 from "@mbse/expressions/SystemV/AMD64";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Repr, Validators } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const { LookupError, NotImplementedError, ValueError } = Errors;
const [C, CV, CText] = [CFamily.Expressions, CFamily.Simplifiers, CFamily.Text];
const [CT, CTText, PP, XT] = [CTypes.Expressions, CTypes.Text, CPreprocessor.Expressions, CppTypes.Expressions];

const store: any = new Builders.Store(); // the C family's dialects, C Types' install before C's: the installs run in any order
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, CTypes.install, CFamily.install, CppFamily.install,
  CppTypes.install, CPreprocessor.install, AMD64.install]) {
  store.transaction(install);
}
for (const define of [CFamily.define, CppFamily.define, AMD64.define, CTypes.define]) store.transaction(define); // C's types and the ABI's, C Types' editions
const AMD64_ = store.Instances.Expressions.SystemV.AMD64; // the ABI a holder names, whose types C leaves to the implementation

/** The term value `spec` writes, the root of `dialect`'s holder, naming `abi`, written in a transaction. */
function term(spec: unknown, dialect: F.Dialect = CT.DIALECT, abi: any = AMD64_): any {
  return F.term(store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => (abi !== null ? e.root(dialect.spec(spec)).abi([abi]) : e.root(dialect.spec(spec))))));
}

const validate = (spec: unknown, bound: string[] = [], dialect: F.Dialect = CT.DIALECT): string[] =>
  dialect.validate(term(spec, dialect), bound);
const text = (spec: unknown, dialect: F.Dialect = CT.DIALECT): string => CTText.ToText(term(spec, dialect));
const union = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name); // a union's branches
const schemas = (name: string): Map<string, any> => new Map(store.registered(name).branches.map((b: any) => [b.name, b.schema]));
const spelled = (word: string) => (i: any) => i.spelling(word); // an identifier, of its spelling
const named = (word: string) => (n: any) => n.identifier(spelled(word)); // a name that is an identifier
const keyword = (word: string) => (s: any) => s.primitivetypespecifier((p: any) => p.keyword(word));
/** A type name, of its specifiers, and its declarator where given. */
const typeName = (specifiers: unknown[], declarator: unknown = null) => (t: any) => t.typeid((i: any) =>
  (declarator === null ? i.specifiers(specifiers) : i.specifiers(specifiers).declarator(declarator)));
const int_ = typeName([keyword("int")]);
const x = C.identifier("x");
/** `word = value`. */
const enumerator = (word: string, value: unknown = null) => (u: any) => u.enumerator(
  (r: any) => (value === null ? r.name(spelled(word)) : r.name(spelled(word)).value(value)));
/** `enum E { items }`. */
const enumeration = (items: unknown[], trailing = false, word = "E") => typeName([(s: any) => s.enumspecifier(
  (e: any) => e.key("enum").name(named(word)).body(true).enumerators(items).trailing_comma(trailing))]);
/** `#ifdef word`, its items, C Types' terms, where an enumeration holds it. */
const ifdef = (word: string, items: unknown[]) => CT.DIALECT.spec((u: any) => u.ifdefdirective((d: any) => d.name(spelled(word)).items(
  items.map((item) => CT.DIALECT.spec(item)))));


<a id="CTY-01"></a>
## CTY-01 · Its union and holder: its kinds, a type name and an enumerator, invalid syntax, and the preprocessor's directives and comment, an enumeration's items; a holder refining the framework's, naming its ABI, round-tripped through JSON

In [ ]:
{
  // Its Term: its kinds, mbse-programs', then invalid syntax, then C Preprocessor's directives and comment, which its
  // install adds, an enumeration's items; C's Term lists its typeid, for a subterm that may be an expression or a type
  assert(same(CT.TAGS, ["typeid", "enumerator"]) && same([...CT.DIALECT.kinds().keys()], [...CT.TAGS, "error", "missing"]));
  assert(same(union(CT.DIALECT.union), [...CT.TAGS, "error", "missing", ...PP.DIRECTIVES, "comment"]));
  assert(CT.DIALECT.name() === "C Types" && CT.DIALECT.tag === "c_types" && union("Expressions.Framework.Term").includes("c_types"));
  const [kinds, held] = [schemas(CT.DIALECT.union), schemas(C.DIALECT.union)];
  assert(held.get("typeid") === kinds.get("typeid") && kinds.get("typeid") === store.registered("Expressions.C.Types.TypeId")); // one schema, in each union
  assert(!held.has("enumerator") && PP.DIRECTIVES.every((tag) => kinds.get(tag) === schemas(PP.DIALECT.union).get(tag)));
  assert(same(CT.DIALECT.reads(), [C.DIALECT, PP.DIALECT]) && C.DIALECT.kindNamed("typeid") === CT.DIALECT.kinds().get("typeid"));
  // Its holder: a reference object refining the framework's Expression, its root narrowed to its Term, naming its ABI.
  const Holder: any = store.registered("Expressions.C.Types.Expression");
  assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(CT.DIALECT.union) && same([...Holder.adjacencies.keys()], ["abi"]));
  const color = enumeration([enumerator("RED", C.literal(1n)), enumerator("GREEN")]);
  const [holder, linked] = store.transaction(() => {
    const holder = store.Expressions.C.Types.Expression((e: any) => e.root(CT.DIALECT.spec(color)).abi([AMD64_]));
    const linked = store.Schemas.OfExpression((s: any) => s.name("cty.color").expression((a: any) => a.insert((e: any) => e.expression(holder))));
    return [holder, linked];
  });
  assert(Validators.Validate(store)(Holder, holder).length === 0 && CT.DIALECT.validate(holder).length === 0);
  assert(F.dialectOf(F.term(holder)) === CT.DIALECT.union && F.kindOf(F.term(holder))[0] === "typeid");
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)) as any).root, holder.root));
  assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, holder)), [true]));
  assert(same([...holder.abi].map((entry: any) => entry.abi), [AMD64_]));
  raises(Error, () => store.transaction(() => // its root holds its own terms
    store.Expressions.C.Types.Expression((e: any) => e.root(CT.DIALECT.spec(C.binary("+", x, 1n))))),
  "'binaryexpression' is not a member of a union value");
}


<a id="CTY-02"></a>
## CTY-02 · Its kinds and helpers: mbse-programs' properties in its order, a type name's parts its helpers, an enumeration's body its terms, the terms they hold C's, and the subterms of C that are its terms

In [ ]:
{
  // Its kinds' properties, as mbse-programs has them, and an enumeration's body: whether it has one, its items, its
  // trailing comma (mbse-programs' `EnumeratorList`, omitted as a wrapper)
  const props = (name: string): string[] => [...store.registered(`Expressions.C.Types.${name}`).properties.keys()];
  assert(same(props("TypeId"), ["specifiers", "declarator"]) && same(props("Enumerator"), ["name", "attributes", "value"]));
  assert(same(props("EnumSpecifier"), ["key", "attributes", "name", "base", "body", "enumerators", "trailing_comma"]));
  assert(same(props("ArrayDeclarator"), ["declarator", "static", "qualifiers", "size", "star", "attributes"]));
  // Its helpers, a type name's parts, which no expression of C holds, each union named after mbse-programs' category
  assert(same(CT.HELPERS, ["DeclSpecifier", "CvQualifier", "PrimitiveTypeSpecifier", "NamedTypeSpecifier", "ClassSpecifier",
    "EnumSpecifier", "PlaceholderTypeSpecifier", "TypeofSpecifier", "AtomicTypeSpecifier", "BitIntSpecifier",
    "IdDeclarator", "PointerDeclarator", "ArrayDeclarator", "FunctionDeclarator", "ParenthesizedDeclarator",
    "ParameterDeclaration", "EllipsisParameter", "StandardAttributeSpecifier", "Attribute", "AlignasSpecifier",
    "GnuAttributeSpecifier", "DeclspecSpecifier"]));
  const kinds = schemas(CT.DIALECT.union);
  assert(CT.HELPERS.every((helper) => !store.registered(`Expressions.C.Types.${helper}`).ref && !kinds.has(helper.toLowerCase())));
  assert(same(union("Expressions.C.Types.Specifier").slice(-4), union("Expressions.C.Types.AttributeSpecifier"))); // its specifiers hold them
  assert(same(union("Expressions.C.Types.Declarator"), ["iddeclarator", "pointerdeclarator", "arraydeclarator", "functiondeclarator",
    "parenthesizeddeclarator"]));
  assert(same(union("Expressions.C.Types.Parameter"), ["parameterdeclaration", "ellipsisparameter"]));
  // The terms a type name holds are C's, and its names C's Name: read through `reads`
  const [CTerm, Name] = [store.registered(C.DIALECT.union), store.registered("Expressions.C.Expressions.Name")];
  const prop = (schema: string, name: string): any => store.registered(`Expressions.C.Types.${schema}`).properties.get(name).schema;
  assert(prop("ArrayDeclarator", "size") === CTerm && prop("TypeofSpecifier", "operand") === CTerm && prop("Enumerator", "value") === CTerm);
  assert(prop("Attribute", "arguments").element === CTerm && prop("NamedTypeSpecifier", "name") === Name && prop("IdDeclarator", "name") === Name);
  assert(prop("Enumerator", "name") === store.registered("Expressions.C.Expressions.Identifier"));
  assert(prop("EnumSpecifier", "enumerators").element === store.registered(CT.DIALECT.union)); // its items, this dialect's terms
  assert(prop("EnumSpecifier", "base") === prop("AtomicTypeSpecifier", "type") && prop("EnumSpecifier", "base") === store.registered(CT.DIALECT.union));
  // C's type subterms are of its Term: a cast's, a compound literal's and a generic association's, each a term
  const Types = store.registered(CT.DIALECT.union);
  assert(["CastExpression", "CompoundLiteralExpression", "GenericAssociation"].every(
    (s) => store.registered(`Expressions.C.Expressions.${s}`).properties.get("type").schema === Types));
  raises(LookupError, () => store.registered("Expressions.C.Expressions.Type"), "no schema registered as 'Expressions.C.Expressions.Type'"); // moved, no alias
  // A type name and an enumerator, as forms: a helper a Part, the terms its parts hold its arguments
  assert(Repr.repr(F.form(term(int_))) === "Form(kind='typeid', attributes={'specifiers': [Part(kind='primitivetypespecifier', "
    + "attributes={'keyword': 'int'}, size=0)]}, arguments=())");
  assert(Repr.repr(F.form(term(enumerator("A", C.literal(1n)))).attributes) === "{'name': Part(kind='identifier', attributes={'spelling': 'A'}, size=0)}");
  const made = F.form(term(enumeration([enumerator("A", C.literal(1n))])));
  assert(made.kind === "typeid" && made.arguments[0] === null && same(made.arguments.slice(1).map((a: any) => F.kindOf(a)[0]), ["enumerator"]));
  // its arguments: its specifier's, an enumeration's fixed type, empty, then its body's items
}


<a id="CTY-03"></a>
## CTY-03 · Its checks: an enumerator needs a name, an identifier's spelling, an enumeration's enumerators within its body, and an enumeration's items of C alone

In [ ]:
{
  assert(same(validate((u: any) => u.enumerator((r: any) => r)), ["an enumerator needs a name"]));
  assert(same(validate(enumerator("1A")), ["'1A' is not an identifier"]) && validate(enumerator("été")).length === 0);
  assert(same(validate(typeName([(s: any) => s.namedtypespecifier((n: any) => n.name(named("2t")))])), ["'2t' is not an identifier"]));
  assert(same(validate(typeName([(s: any) => s.enumspecifier((e: any) => e.key("enum").enumerators([enumerator("A")]))])), [
    "an EnumSpecifier with enumerators has a body"]));
  assert(same(validate(typeName([(s: any) => s.enumspecifier((e: any) => e.key("enum").trailing_comma(true))])), [
    "an EnumSpecifier with enumerators has a body"]));
  assert(validate(typeName([(s: any) => s.enumspecifier((e: any) => e.key("enum").body(true))])).length === 0); // `enum {}`
  // Its terms' own checks: a literal's, within an enumerator's value; a value's references are its scope's
  assert(same(validate(enumerator("A", C.literal(1n, "q"))), ["value: a literal's suffix cannot be 'q'"]));
  assert(same(validate(enumerator("B", x)), ["value: variable 'x' is not bound"]) && validate(enumerator("B", x), ["x"]).length === 0);
  // An enumeration's items are its own dialect's terms: a directive's items, any dialect's by schema, are C Types' within C's
  const cppItem = XT.DIALECT.spec((u: any) => u.enumerator((r: any) => r.name(spelled("B"))));
  const directive = CT.DIALECT.spec((u: any) => u.ifdefdirective((d: any) => d.name(spelled("B_")).items([cppItem])));
  assert(same(validate(enumeration([directive])), [
    "specifiers 0 argument 0: argument 0: a Cpp Types enumerator is not a C Types term"]));
  assert(validate(enumeration([ifdef("B_", [enumerator("B")])])).length === 0);
}


<a id="CTY-04"></a>
## CTY-04 · What it binds: nothing; the references an enumerator's value or a type name's terms hold are their scope's

In [ ]:
{
  assert(same([...SY.free(term(enumeration([enumerator("A", C.literal(1n)), enumerator("B", C.identifier("A"))])), CT.DIALECT)], ["A"]));
  const sized = typeName([keyword("int")], (d: any) => d.arraydeclarator((a: any) => a.size(C.binary("+", x, 1n)))); // int[x + 1]
  assert(same([...SY.free(term(sized), CT.DIALECT)], ["x"]));
  assert(same(validate(sized), ["declarator size: left: variable 'x' is not bound"]) && validate(sized, ["x"]).length === 0);
  const typeof_ = typeName([(s: any) => s.typeofspecifier((t: any) => t.keyword("typeof").operand(x))]); // typeof(x)
  assert(same([...SY.free(term(typeof_), CT.DIALECT)], ["x"]) && !store.registered(CT.DIALECT.union).branches.some((b: any) => F.isBinder(b.schema)));
}


<a id="CTY-05"></a>
## CTY-05 · Within C's terms: a cast's, a compound literal's and a generic association's type, and sizeof's, a C Types term C reads, validates, infers, simplifies and prints; written by C's writers

In [ ]:
{
  const uint8 = typeName([(s: any) => s.namedtypespecifier((n: any) => n.name(named("uint8_t")))]);
  const cast = term(C.cast(uint8, C.binary("+", x, 1n)), C.DIALECT); // (uint8_t)(x + 1)
  assert(F.form(cast).kind === "castexpression" && same(F.form(cast).arguments.map((a: any) => F.dialectOf(a)), [CT.DIALECT.union, C.DIALECT.union]));
  assert(F.kindOf(F.form(cast).arguments[0])[0] === "typeid" && C.DIALECT.kind(F.form(cast).arguments[0])[2] === CT.DIALECT.kinds().get("typeid"));
  assert(C.DIALECT.validate(cast, ["x"]).length === 0 && CText.ToText(cast) === "(uint8_t)(x + 1)" && equal(CV.SIMPLIFIER.rules.evaluate(cast, new CV.Scope({ x: 255n })).value, 0n));
  assert(CText.ToText(term(CV.OfAny(cast, { x: 255n }).residual, C.DIALECT)) === "(uint8_t)256"); // a uint8_t has no literal: x folds
  assert(C.DIALECT.infer(cast, { x: D.Anything }).name() === "uint8_t" && F.same(term(C.DIALECT.make(F.form(cast)), C.DIALECT), cast));
  assert(same([...SY.free(cast, C.DIALECT)], ["x"])); // the type holds no reference: the operand's alone
  const cases: [unknown, string][] = [
    [(u: any) => u.sizeofexpression((r: any) => r.operand(int_)), "sizeof(int)"],
    [(u: any) => u.compoundliteralexpression((r: any) => r.type(int_).initializer((i: any) => i.items([C.literal(1n)]))), "(int){1}"],
    [(u: any) => u.genericselection((r: any) => r.controlling(x).associations([(a: any) => a.type(int_).value(C.literal(1n)),
      (a: any) => a.value(C.literal(0n))])), "_Generic(x, int: 1, default: 0)"],
    [(u: any) => u.sizeofexpression((r: any) => r.operand(enumeration([enumerator("A")]))), "sizeof(enum E {\n    A\n})"],
  ];
  for (const [spec, source] of cases) {
    assert(C.DIALECT.validate(term(spec, C.DIALECT), ["x"]).length === 0 && CText.ToText(term(spec, C.DIALECT)) === source, source);
  }
  const generic = F.form(term((u: any) => u.genericselection((r: any) => r.controlling(x).associations([(a: any) => a.value(C.literal(0n))])), C.DIALECT));
  assert(generic.arguments[1] === null && F.kindOf(generic.arguments[2])[0] === "literal"); // an association: a pair, its type absent
  // From its own holder, its terms' C terms are read through C
  const chars = typeName([keyword("char")], (d: any) => d.arraydeclarator((a: any) => a.size(C.literal(4n)))); // char[4]
  assert(CT.DIALECT.infer(term(int_)) === D.Anything && CTText.ToText(term(chars)) === "char[4]" && validate(chars).length === 0);
}


<a id="CTY-06"></a>
## CTY-06 · Invalid syntax: error and missing in its Term and among an enumeration's items, validated, printed as written, refused by name

In [ ]:
{
  const skipped = (u: any) => u.error((r: any) => r.text("int int"));
  const expected = (u: any) => u.missing((r: any) => r.kind("type"));
  assert(same(validate(skipped), ["invalid syntax 'int int'"]) && same(validate(expected), ["missing type"]));
  assert(text(skipped) === "int int" && text(expected) === "");
  assert(same(C.DIALECT.validate(term((u: any) => u.castexpression((r: any) => r.type(skipped).operand(C.literal(1n))), C.DIALECT)), [
    "type: invalid syntax 'int int'"]));
  raises(NotEvaluated, () => CV.OfAny(term((u: any) => u.castexpression((r: any) => r.type(skipped).operand(C.literal(1n))), C.DIALECT)).evaluate(),
    "C Types error is not simplified"); // a cast reads its type: invalid syntax there stays, named
  raises(ValueError, () => text((u: any) => u.error((r: any) => r)), "C Types error has no text to print");
  const broken = enumeration([enumerator("A"), (u: any) => u.error((r: any) => r.text("B C"))]);
  assert(same(validate(broken), ["specifiers 0 argument 1: invalid syntax 'B C'"]) && text(broken) === "enum E {\n    A\n    B C\n}");
}


<a id="CTY-07"></a>
## CTY-07 · Its editions: C's three, each by its designation and common name, each with both its kinds

In [ ]:
{
  const types = store.Instances.Expressions.C.Types;
  assert(same([...FS.standards(types).keys()], ["ISO/IEC 9899:2011", "ISO/IEC 9899:2018", "ISO/IEC 9899:2024"]));
  assert(equal([...FS.standards(types).values()].map((s) => FS.tags(s)), [
    ["ISO/IEC 9899:2011", "C11"], ["ISO/IEC 9899:2018", "C17"], ["ISO/IEC 9899:2024", "C23"]]));
  assert(equal(FS.properties(FS.find(types, "C23")!), map({ bodies: ["org.iso", "ch.iec"], number: "9899", revision: ["year", 2024n],
    names: ["C23"] })));
  assert(CT.SINCE.size === 0 && [...FS.standards(types).values()].every((edition) => same(FS.kinds(edition).map((s: any) => s.name),
    ["Expressions.C.Types.TypeId", "Expressions.C.Types.Enumerator"])));
  const own = schemas(CT.DIALECT.union);
  assert([...own].every(([tag, schema]) => same([...FS.of(types, schema).values()].map((s) => FS.tags(s)[FS.tags(s).length - 1]),
    CT.TAGS.includes(tag) ? ["C11", "C17", "C23"] : []))); // invalid syntax, and the preprocessor's kinds, in none of its editions
  assert(store.transaction(CTypes.define)[0].get("C17") === FS.find(types, "C17")); // run again, it writes nothing
  assert(Validators.Validate(store)(store.registered("Expressions.Framework.Standard"), FS.find(types, "C11")).length === 0);
}


<a id="CTY-08"></a>
## CTY-08 · Printing: each type name as mbse-programs' printer writes it, an enumeration's body on lines of its own, its directives at their line's start

In [ ]:
{
  const pointer = (inner: unknown = null) => (d: any) => d.pointerdeclarator((p: any) => (inner === null ? p : p.declarator(inner)));
  const cases: [unknown, string][] = [
    [int_, "int"], [typeName([keyword("unsigned"), keyword("long")]), "unsigned long"],
    [typeName([(s: any) => s.cvqualifier((q: any) => q.keyword("const")), keyword("char")], pointer()), "const char*"],
    [typeName([keyword("int")], (d: any) => d.arraydeclarator((a: any) => a.declarator(pointer()).size(C.literal(4n)))), "int(*)[4]"],
    [typeName([keyword("int")], (d: any) => d.functiondeclarator((f: any) => f.declarator((p: any) => p.parenthesizeddeclarator(
      (q: any) => q.declarator(pointer()))).parameters([(p: any) => p.parameterdeclaration((q: any) => q.specifiers([keyword("int")])),
      (p: any) => p.ellipsisparameter((e: any) => e)]))), "int(*)(int, ...)"],
    [typeName([(s: any) => s.classspecifier((c: any) => c.key("struct").name(named("S")).body(true))]), "struct S {}"],
    [typeName([(s: any) => s.enumspecifier((e: any) => e.key("enum").name(named("E")))]), "enum E"],
    [typeName([(s: any) => s.enumspecifier((e: any) => e.key("enum").body(true))]), "enum {}"],
    [typeName([(s: any) => s.enumspecifier((e: any) => e.key("enum").name(named("E")).base(typeName([keyword("char")])).body(true).enumerators([
      enumerator("A")]))]), "enum E : char {\n    A\n}"],
    [enumeration([enumerator("A", C.literal(1n)), enumerator("B")]), "enum E {\n    A = 1,\n    B\n}"],
    [enumeration([enumerator("A"), enumerator("B")], true), "enum E {\n    A,\n    B,\n}"],
    [enumeration([enumerator("A", C.binary("+", C.literal(1n), C.literal(2n))), (u: any) => u.comment((c: any) => c.text(" last").trailing(true))]),
      "enum E {\n    A = 1 + 2 // last\n}"],
    [enumeration([(u: any) => u.comment((c: any) => c.text(" first ").block(true)), enumerator("A"),
      ifdef("WIDE", [enumerator("B"), enumerator("C")]), enumerator("D")]),
      "enum E {\n    /* first */\n    A,\n#ifdef WIDE\n    B,\n    C,\n#endif\n    D\n}"],
    [enumeration([enumerator("A"), ifdef("B_", [enumerator("B")])]), "enum E {\n    A\n#ifdef B_\n    B,\n#endif\n}"],
    // the last enumerator of its body's own without a comma, as mbse-programs writes it
    [enumeration([enumerator("A", (u: any) => u.sizeofexpression((r: any) => r.operand(enumeration([enumerator("B")], false, "F"))))]),
      "enum E {\n    A = sizeof(enum F {\n        B\n    })\n}"], // an enumeration within, one level deeper
    [enumerator("A", C.literal(1n)), "A = 1"], [(u: any) => u.comment((c: any) => c.text(" note")), "// note"],
  ];
  for (const [spec, source] of cases) assert(text(spec) === source, `${source} != ${text(spec)}`);
  assert(CTText.ToText(term(C.cast(int_, x), C.DIALECT)) === CText.ToText(term(C.cast(int_, x), C.DIALECT))
    && CText.ToText(term(C.cast(int_, x), C.DIALECT)) === "(int)x"); // one printer
}


<a id="CTY-09"></a>
## CTY-09 · Simplification: none folds (Q11); each of its kinds stays in the residual, named, wherever it stands; a cast reads its type rather than fold it

In [ ]:
{
  for (const [spec, tag] of [[int_, "typeid"], [enumerator("A"), "enumerator"]] as [unknown, string][]) { // C's evaluator, which reads C Types' terms
    raises(NotEvaluated, () => CV.OfAny(term(spec), {}).evaluate(), `C Types ${tag} is not simplified`);
  }
  raises(NotEvaluated, () => CV.OfAny(term((u: any) => u.sizeofexpression((r: any) => r.operand(int_)), C.DIALECT)).evaluate(),
    "C sizeofexpression is not simplified"); // sizeof is refused, not its type
  assert(equal(CV.SIMPLIFIER.rules.evaluate(term(C.cast(typeName([keyword("unsigned"), keyword("char")]), 300n), C.DIALECT), new CV.Scope()).value, 44n)); // the cast converts
  raises(NotEvaluated, () => CV.OfAny(term((u: any) => u.typeid((i: any) => i.specifiers([keyword("int")])), C.DIALECT)).evaluate(),
    "C Types typeid is not simplified"); // a type where an expression may stand, alone
}
